# Prompt → SDXL → ASCII Art Pipeline

**Workflow:**
1. Run **Cell 1** once to install all dependencies.
2. Run **Cell 2** once to load SDXL (~6 GB download on first run).
3. **Edit your prompt in Cell 3 and run it** — image is generated, ASCII rendered, saved to `output/`.
4. Run **Cell 4** to preview results inline in the notebook.
5. *(Optional)* Use **Cell 5** to run a batch of prompts.

> Auto-detects GPU (CUDA), Apple Silicon (MPS), or CPU.

In [ ]:
# Cell 1 — Install dependencies (run once, safe to re-run)
import subprocess, sys, importlib

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip('pillow')
pip('diffusers>=0.27.0', 'transformers>=4.40.0', 'accelerate>=0.29.0', 'safetensors')

# Install CPU torch if torch is not already present
if importlib.util.find_spec('torch') is None:
    pip('torch', 'torchvision',
        '--index-url', 'https://download.pytorch.org/whl/cpu')

print('All dependencies ready.')


In [ ]:
# Cell 2 — Load SDXL model (run once per kernel session)
import torch
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

# Auto-detect device
if torch.cuda.is_available():
    DEVICE, DTYPE = 'cuda', torch.float16
    print('GPU:', torch.cuda.get_device_name(0))
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE, DTYPE = 'mps', torch.float16
    print('Apple Silicon MPS')
else:
    DEVICE, DTYPE = 'cpu', torch.float32
    print('CPU — generation will be slow (~5-15 min per image)')

MODEL_ID = 'stabilityai/stable-diffusion-xl-base-1.0'
print('Loading', MODEL_ID, '...')

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    use_safetensors=True,
    variant='fp16' if DTYPE == torch.float16 else None,
)

# DPM++ 2M Karras scheduler — good quality at 25-30 steps
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, use_karras_sigmas=True)

# Memory optimisations
if DEVICE == 'cuda':
    try:
        pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pass
    pipe.enable_vae_slicing()
    pipe = pipe.to(DEVICE)
elif DEVICE == 'mps':
    pipe = pipe.to(DEVICE)
else:
    # CPU: sequential offload avoids OOM on 8 GB RAM machines
    pipe.enable_sequential_cpu_offload()

print('Model loaded and ready.')


In [ ]:
# Cell 3 — EDIT YOUR PROMPT, then run this cell
import sys, os, re, time
from pathlib import Path
from PIL import Image

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PROMPT — change this to anything you want
PROMPT = 'a glowing volcano erupting at night, molten lava rivers, dramatic lightning, epic cinematic'
NEGATIVE_PROMPT = 'blurry, low quality, watermark, text, oversaturated, cartoon'

# Generation settings
NUM_STEPS       = 30      # 20-40 recommended
GUIDANCE_SCALE  = 7.5
IMAGE_WIDTH     = 1024    # SDXL native resolution
IMAGE_HEIGHT    = 1024
SEED            = None    # set an int for reproducible results, e.g. 42

# ASCII render settings
ASCII_WIDTH      = 160    # character columns
ASPECT_RATIO     = 0.50
PALETTE          = 'dots' # 'dots' | 'clean' | 'detailed' | 'pixel'
FONT_SIZE_HTML   = 9
FONT_SIZE_MOTION = 10
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Ensure customRenderer is importable
NB_DIR = Path(os.getcwd())
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))
from customRenderer import AsciiRenderer

OUTPUT_DIR = NB_DIR / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

# ── 1. Generate image ────────────────────────────────────────────────────────
print(f'Generating image for prompt: {PROMPT!r}')
t0 = time.time()

generator = (torch.Generator(device=DEVICE).manual_seed(SEED)
             if SEED is not None else None)

result = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    num_inference_steps=NUM_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    width=IMAGE_WIDTH,
    height=IMAGE_HEIGHT,
    generator=generator,
)
generated_img = result.images[0]
elapsed = time.time() - t0
print(f'Image generated in {elapsed:.1f}s')

# Save raw PNG
slug = re.sub(r'[^a-z0-9]+', '_', PROMPT[:40].lower()).strip('_')
raw_img_path = OUTPUT_DIR / f'{slug}_raw.png'
generated_img.save(raw_img_path)
print(f'Raw image saved: {raw_img_path}')

# ── 2. ASCII render ──────────────────────────────────────────────────────────
print('Rendering ASCII art...')
renderer = AsciiRenderer(
    width=ASCII_WIDTH,
    aspect_ratio=ASPECT_RATIO,
    palette=PALETTE,
    contrast=1.35,
    saturation=1.4,
    brightness=1.0,
    edge_detection=False,
)

# Static full-color HTML
color_html_path = OUTPUT_DIR / f'{slug}_color.html'
renderer.render_to_html(generated_img,
                        output_file=color_html_path,
                        font_size=FONT_SIZE_HTML)
print(f'Color HTML  -> {color_html_path}')

# Animated motion HTML
motion_html_path = OUTPUT_DIR / f'{slug}_motion.html'
renderer.render_motion_html(generated_img,
                            output_file=motion_html_path,
                            font_size=FONT_SIZE_MOTION)
print(f'Motion HTML -> {motion_html_path}')

print('Done! Open the HTML files in your browser.')


In [ ]:
# Cell 4 — Preview the generated image inline
from IPython.display import display, HTML, Image as IPyImage

print(f'Prompt: {PROMPT}')
display(IPyImage(filename=str(raw_img_path), width=512))

display(HTML(
    '<div style="font-family:monospace;line-height:2;margin-top:8px">'
    '<b>Outputs:</b><br>'
    f'<a href="{color_html_path}" target="_blank">Color HTML (static)</a><br>'
    f'<a href="{motion_html_path}" target="_blank">Motion HTML (animated)</a>'
    '</div>'
))


In [ ]:
# Cell 5 (optional) — Batch: run multiple prompts
import re, time
from pathlib import Path
from customRenderer import AsciiRenderer

BATCH_PROMPTS = [
    'a cyberpunk city at night, neon reflections, rain, ultra detailed',
    'deep ocean bioluminescent jellyfish, dark water, ethereal glow',
    'ancient forest golden hour, shafts of light, misty atmosphere',
]

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

r = AsciiRenderer(width=160, aspect_ratio=0.50, palette='dots',
                  contrast=1.35, saturation=1.4)

for prompt in BATCH_PROMPTS:
    print(f'Generating: {prompt!r}')
    t0 = time.time()
    img = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=NUM_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        width=IMAGE_WIDTH,
        height=IMAGE_HEIGHT,
    ).images[0]
    elapsed = time.time() - t0
    print(f'  Generated in {elapsed:.1f}s')

    slug = re.sub(r'[^a-z0-9]+', '_', prompt[:40].lower()).strip('_')
    img.save(OUTPUT_DIR / f'{slug}_raw.png')
    r.render_to_html(img,
                     output_file=OUTPUT_DIR / f'{slug}_color.html',
                     font_size=9)
    r.render_motion_html(img,
                         output_file=OUTPUT_DIR / f'{slug}_motion.html',
                         font_size=10)
    print(f'  Saved output/{slug}_*.html')

print('Batch complete!')
